# Classification of Ice and Open Water in Nizhnesvirsky Lower Bay using Sentinel-1 IW Product
### The ________ is a vital part of the Yukon River system characterized by a dynamic freeze-thaw cycle. Understanding the extent of ice coverage and open water in this region is crucial for various applications, including navigation, ecosystem monitoring, and climate research. In this Jupyter notebook, I leverage satellite imagery from the Sentinel-1 Interferometric Wide (IW) product, along with remote sensing principles, to classify ice and open water over a period of 8 months, from September 2022 to May 2023.

### The Sentinel-1 satellite offers reliable and frequent radar imaging capabilities, making it ideal for monitoring ice and water bodies. The Sentinel-1 IW product provides Synthetic Aperture Radar (SAR) data, which is unaffected by weather conditions, daylight, or cloud cover. SAR data measures the backscattered electromagnetic waves, allowing us to distinguish between different surface types.

In [1]:
import datetime
import getpass

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import geopandas as gpd
import json
from ipyleaflet import Map, GeoJSON, basemaps

from sentinelhub import (
    SHConfig,
    Geometry,
    DataCollection,
    MimeType,
    SentinelHubDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
)

In [2]:
# Only run this cell if you have not created a configuration.

config = SHConfig()
config.sh_client_id = "sh-727a5dbe-c70a-46cc-bc9d-1d43c629e6d3"
config.sh_client_secret = "rWViN385vwzbVJqr9dV8l6JTDyv227mP"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save()

In [11]:
SHAPE_PATH = "../data/raw/yukon_river.geojson"
river_gdf = gpd.read_file(SHAPE_PATH)

# Convert to EPSG 3035 for metric resolution calculations
river_gdf = river_gdf.to_crs("EPSG:32604")

resolution = 20

data_geojson = json.load(open(SHAPE_PATH, "r"))

# Set center and zoom level
center = [64.727262, -156.970460]
zoom = 9

# Build and display interactive map
m = Map(basemap=basemaps.OpenStreetMap.Mapnik, center=center, zoom=zoom)
geo_json = GeoJSON(data=data_geojson)
m.add_layer(geo_json)
m

Map(center=[64.727262, -156.97046], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [3]:
evalscript_sar = """
  function setup() {
    return {
      input: ["VV", "dataMask"],
      output: { bands: 2, sampleType: "FLOAT32"}                  // Refer to Note 1
    }
  }


// visualizes decibels from -20 to +10
function toDb(linear) {
  var log = 10 * Math.log(linear) / Math.LN10
  return Math.max(0, (log + 20) / 30)
}

function evaluatePixel(sample) {
  var VV = sample.VV;

  return [toDb(VV),sample.dataMask];
}
"""


def get_sar_request(time_interval):
    return SentinelHubRequest(
        evalscript=evalscript_sar,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL1_IW.define_from(
                    "s1iw", service_url=config.sh_base_url
                ),
                time_interval=time_interval,
                other_args={
                    "dataFilter": {
                        "resolution": "HIGH",
                        "mosaickingOrder": "mostRecent",
                        "orbitDirection": "ASCENDING",
                    },
                    "processing": {
                        "backCoeff": "SIGMA0_ELLIPSOID",
                        "orthorectify": True,
                        "demInstance": "COPERNICUS",
                        "speckleFilter": {
                            "type": "LEE",
                            "windowSizeX": 3,
                            "windowSizeY": 3,
                        },
                    },
                },
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        geometry=Geometry(river_gdf.geometry.values[0], crs=river_gdf.crs),
        resolution=[20, 20],  # Refer to Note 2
        config=config,
        data_folder="./results",
    )

In [4]:
start = datetime.datetime(2022, 9, 1)
end = datetime.datetime(2023, 4, 30)
n_chunks = 17
tdelta = (end - start) / n_chunks
edges = [(start + i * tdelta).date().isoformat() for i in range(n_chunks)]
slots = [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]

print("Monthly time windows:\n")
for slot in slots:
    print(slot)

Monthly time windows:

('2022-09-01', '2022-09-15')
('2022-09-15', '2022-09-29')
('2022-09-29', '2022-10-13')
('2022-10-13', '2022-10-27')
('2022-10-27', '2022-11-10')
('2022-11-10', '2022-11-25')
('2022-11-25', '2022-12-09')
('2022-12-09', '2022-12-23')
('2022-12-23', '2023-01-06')
('2023-01-06', '2023-01-20')
('2023-01-20', '2023-02-03')
('2023-02-03', '2023-02-18')
('2023-02-18', '2023-03-04')
('2023-03-04', '2023-03-18')
('2023-03-18', '2023-04-01')
('2023-04-01', '2023-04-15')


In [5]:
# create a list of requests
list_of_requests = [get_sar_request(slot) for slot in slots]
list_of_requests = [request.download_list[0] for request in list_of_requests]

# download data with multiple threads
data = SentinelHubDownloadClient(config=config).download(
    list_of_requests, max_threads=5
)

NameError: name 'river_gdf' is not defined

In [11]:
SHAPE_PATH = "./data/Nizhnesvirsky_lower_bay.geojson"
river_gdf = gpd.read_file(SHAPE_PATH)

# Convert to EPSG 3035
river_gdf = river_gdf.to_crs("EPSG:3035")
# Geometry of an entire area
resolution = 20

data = json.load(open(SHAPE_PATH, "r"))

# Set center and zoom level
center = [60.78, 33.68]
zoom = 12

# Add OSM background
m = Map(basemap=basemaps.OpenStreetMap.Mapnik, center=center, zoom=zoom)

# Add geojson data
geo_json = GeoJSON(data=data)
m.add_layer(geo_json)

# Display
m

DriverError: ./data/Nizhnesvirsky_lower_bay.geojson: No such file or directory